# QA to compare with MEM1

In [1]:
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import pdb
import numpy as np
import plotly.graph_objects as go
import json

def get_df_from_train_gen(file: str):
    raw_jl = open(file, 'r', encoding='utf-8').read()


    json_list = []
    for i, raw_json in enumerate(raw_jl.strip().split("\n}")):
        if raw_json.strip():
            json_list.append(json.loads(raw_json + "\n}"))
    df = pd.DataFrame(json_list)
    return df

def deduplicate_df(dfraw):
    odf = pd.concat([pd.DataFrame(dfraw.iloc[i].to_dict()) for i in range(len(dfraw))]) # concatenate all the evaluated batches. some questions may be duplicated across batches
    odf['questions']=odf.apply(lambda x: x.full_trajectory_strings.split('Answer the following questions: ')[1].split('\n')[0], axis=1) # extract questions 
    keep_uid_per_q = ( # one traj_uid per question (first seen)
        odf.groupby('questions')['traj_uid']
        .transform('first'))            # the chosen uid for that question
    odf_deduplicated = odf[odf['traj_uid'] == keep_uid_per_q]
    assert len(odf_deduplicated.groupby('traj_uid').size()) == len(odf_deduplicated.groupby('questions').size())
    return odf_deduplicated

eval_folder_path = "/nas/ucb/jbjorner3/dev/optimal-explorer-dev/verl-agent/checkpoints/verl_agent_alfworld/"

REPORTED_OBJS = [2,8,16]
ALL_NUM_OBJS = [1, 2, 4, 8, 16]
OBJS_TO_X = {1: 1, 2: 2, 4: 3.5, 8: 5.5, 16: 8}



In [4]:
#  load full eval set
mem1_objectives_dfs ={}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_TrueGRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    mem1_objectives_dfs[num_objs] = deduplicate_df(dfraw)

In [4]:
len(mem1_objectives_dfs[1])

160332

In [5]:
temperature = '1.0'
length_penalty = '0.1'
abbel_objectives_dfs_penalty ={}
for num_objs in [1, 2, 4, 8, 16]:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT_4/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    abbel_objectives_dfs_penalty[num_objs] = deduplicate_df(dfraw)

In [6]:
temperature = '1.0'
length_penalty = '0.05'
abbel_objectives_dfs_penalty05 = {}
for num_objs in [1,2,4,8,16]:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT_4/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    abbel_objectives_dfs_penalty05[num_objs] = deduplicate_df(dfraw)

In [5]:
# temperature = '1.0'
# length_penalty = '0.0'
# abbel_objectives_dfs ={}
# for num_objs in [1,2,4,8,16]:
#     dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
#     abbel_objectives_dfs[num_objs] = deduplicate_df(dfraw)

In [7]:
temperature = '0.01'
length_penalty = '0.0'
abbel_objectives_dfs01 ={}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    abbel_objectives_dfs01[num_objs] = deduplicate_df(dfraw)

# Random subset evals

In [4]:
mem1_objectives_dfs ={}
for num_objs in [1, 2, 4, 8, 16]:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_TrueGRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference/train_gen.txt")
    odf = pd.DataFrame(dfraw.iloc[0].to_dict())
    mem1_objectives_dfs[num_objs] = odf

In [2]:
temperature = '0.01'
abbel_objectives_dfs01 ={}
for num_objs in [1, 2, 4, 8, 16]:
    dfraw = get_df_from_train_gen(f"/nas/ucb/dayan/optimal-explorer-dev/verl-agent/checkpoints/verl_agent_alfworld/nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_0.0GRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference/train_gen.txt")
    odf = pd.DataFrame(dfraw.iloc[0].to_dict())
    abbel_objectives_dfs01[num_objs] = odf

In [10]:
temperature = '1.0'
length_penalty = '0.1'
abbel_objectives_dfs_penalty ={}
for num_objs in [1, 2, 4, 8, 16]:
    dfraw = get_df_from_train_gen(f"/nas/ucb/jbjorner3/dev/optimal-explorer-dev/verl-agent/checkpoints/temp/nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_128sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT_3/global_step_200_objectives_{str(num_objs)}_inference/train_gen.txt")
    odf = pd.DataFrame(dfraw.iloc[0].to_dict())
    abbel_objectives_dfs_penalty[num_objs] = odf

# Qualitative Look At Trajectories

In [5]:
for i, traj in abbel_objectives_dfs01[16].groupby('traj_uid'):
    if traj.rewards.max() > 3:
        break
print(traj.sort_values(by='step').input_ids_str.iloc[5])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Global Instruction: <instruction>You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: Which town is home to the University of Ulster?; Flourine, Bromine, Iodine and Chlorine are all what type of elements?; Which comic chat show host was played by Caroline A'Herne?; Its capital is Valverde 

In [9]:
for i, traj in mem1_objectives_dfs[16].groupby('traj_uid'):
    break
print(traj.sort_values(by='step').input_ids_str.iloc[5])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
You will answer multiple complex questions using iterative reasoning, summarization, and web search.

At each step, you will see the questions, a cumulative summary of relevant information, the current search query, and search results (except in the first step, where only the questions are provided). Your task is to:

1. Perform reasoning and update a cumulative, concise summary within <think> ... </think>. This acts as persistent memory and must include all essential information from previous <think> and <information> tags.

2. Then choose one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The a

In [13]:
count = 0
for i, traj_df in abbel_objectives_dfs_penalty[1].groupby('traj_uid'):
    if count == 5:
        print(traj_df.sort_values(by='step').input_ids_str.iloc[5])
        break
    count += 1


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Global Instruction: <instruction>You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: What is the capital of Kuwait?</instruction>
Your current belief state: <belief>Kuwait City</belief>
Your last action: <search>capital of Kuwait</search>
Environment feedback: <environment><hint>You have 

# Getting Token Counts

In [8]:
# instantiate tokenizer
from verl.utils import hf_tokenizer
from verl.utils.fs import copy_to_local

MODEL_PATH='qwen/qwen2.5-7b-instruct'
local_path = copy_to_local(MODEL_PATH, use_shm=False)
tokenizer = hf_tokenizer(local_path, trust_remote_code=False, is_instruct_model=True)

In [68]:
import re

def extract_tag_text(text, tag):
    """Extract first instance of text between <tag></tag> tags"""
    if f"<{tag}>" in text and f"</{tag}>" in text:
        pattern = f'<{tag}>(.*?)</{tag}>'
        matches = re.findall(pattern, text, re.DOTALL)
        tagtext = matches[0].strip()
        subtracted_text = text.replace(f"<{tag}>{tagtext}</{tag}>", "")
    elif f"<{tag}>" in text and (tag in ["information", "environment"]) and (text[-len("\nassistant\n"):] == "\nassistant\n"): # search results got cut off and it goes straight to assistant
        print(f"missing </{tag}>")
        tagtext = text.split(f"<{tag}>",1)[1][:-len("\nassistant\n")] 
        subtracted_text = text.replace(f"<{tag}>{tagtext}","")
    elif (f"<{tag}>" not in text) or (f"</{tag}>" not in text) and (tag == 'think') and '<search>' in text: # malformed internal state for MEM1
        print(f"missing opening and/or closing think tags")
        tagtext = text.split(f"<search>",1)[0] #MEM1 treats everything before <search> as the internal state
        subtracted_text = text.replace(tagtext, "")
    else:
        import pdb; pdb.set_trace()
        return "", text
    
    return tagtext, subtracted_text


def compute_peak_token_ABBEL(traj_df, tokenizer,memory_only=False) -> float:
    if traj_df.iloc[-1].rewards == 0:
        return None
    peak_count = 0
    for i,row in traj_df.iterrows():
        if row.step == 0:
            input_str, input_belief = "", ""
        else:
            input_str = row.input_ids_str.split('</instruction>\n', 1)[1] # remove initial instructions/system prompt as done in Zhou et al. MEM1 evals
            input_belief, input_str_extracted = extract_tag_text(input_str, 'belief')
            if row['info']['action_or_belief'] == 1: # belief update step
                input_action, input_str_extracted = extract_tag_text(input_str_extracted, 'search')
                input_observation, _ = extract_tag_text(input_str_extracted, 'environment')
                if '</hint>' in input_observation:
                    if len(input_observation.split('</hint>\n\n')) < 2:
                        import pdb; pdb.set_trace()
                    input_observation = input_observation.split('</hint>\n\n')[1]
                if len(input_observation) == 0:
                    import pdb; pdb.set_trace()
                input_str = input_belief + input_action + input_observation
            else:
                input_str = input_belief  
        output_str = row.response_ids_str
        if memory_only:
            peak_count = max(peak_count, len(tokenizer.encode(input_belief)))
        else:
            peak_count = max(peak_count, len(tokenizer.encode(input_str + output_str)))

    if peak_count == 0:
        return None

    return peak_count

def compute_peak_token_MEM1(traj_df, tokenizer,memory_only=False) -> float:
    if traj_df.iloc[-1].rewards == 0:
        return None
    peak_count = 0
    for i,row in traj_df.iterrows():
        if row.step == 0:
            input_str, input_IS = "", ""
        else:
            input_str = row.input_ids_str.split('\n\nassistant\n', 1)[1] # remove initial instructions/system prompt as done in Zhou et al. MEM1evals
            input_IS, input_str_extracted = extract_tag_text(input_str, 'think') # previous internal state
            input_action, input_str_extracted = extract_tag_text(input_str_extracted, 'search')
            input_observation, _ = extract_tag_text(input_str_extracted, 'information')
            if len(input_observation) == 0:
                import pdb; pdb.set_trace()
            if '</hint>' in input_observation:
                input_observation = input_observation.split('</hint>\n\n')[1]
            input_str = input_IS + input_action + input_observation
        output_str = row.response_ids_str
        if memory_only:
            peak_count = max(peak_count, len(tokenizer.encode(input_IS)))
        else:
            peak_count = max(peak_count, len(tokenizer.encode(input_str + output_str)))

    if peak_count == 0:
        return None
    return peak_count


def compute_peak_token_all_trajectories(df, framework, tokenizer, memory_only=False) -> float:
# Create a dictionary of dataframes, one for each unique traj_uid
    all_peak_tokens = []
    for traj_uid, group in df.groupby('traj_uid'):
        traj_df = group.reset_index(drop=True).sort_values(by='step')
        if framework == 'ABBEL':
            all_peak_tokens.append(compute_peak_token_ABBEL(traj_df, tokenizer, memory_only))
        elif framework == 'MEM1':
            all_peak_tokens.append(compute_peak_token_MEM1(traj_df, tokenizer, memory_only))
      # filter out None
    all_peak_tokens = [x for x in all_peak_tokens if x is not None]
    return all_peak_tokens


In [69]:
def compute_scores(objectives_dfs):
    all_scores = []
    for num_objs in ALL_NUM_OBJS:
        traj_rewards = objectives_dfs[num_objs].groupby('traj_uid').rewards.max()
        all_scores.append(traj_rewards)
    return all_scores

def compute_tokens(objectives_dfs, framework, tokenizer,memory_only=False):
    all_tokens = []
    for num_objs in ALL_NUM_OBJS:
        peak_tokens = compute_peak_token_all_trajectories(objectives_dfs[num_objs], framework, tokenizer, memory_only)
        all_tokens.append(peak_tokens)
        print("peak tokens for ", num_objs, "objs complete")
    return all_tokens

def compute_tokens_parallel(objectives_dfs, framework, memory_only=False):
    all_tokens = []
    thread_tokenizer = hf_tokenizer(local_path, trust_remote_code=False, is_instruct_model=True)
    for num_objs in ALL_NUM_OBJS:
        peak_tokens = compute_peak_token_all_trajectories(objectives_dfs[num_objs], framework, thread_tokenizer, memory_only)
        all_tokens.append(peak_tokens)
        print("peak tokens for ", num_objs, "objs: ", peak_tokens)
    return all_tokens

In [83]:
mtokens = compute_tokens(mem1_objectives_dfs, 'MEM1', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
missing </information>
missing opening and/or closing think tags
peak tokens for  4 objs complete
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
peak tokens for  8 objs complete
peak 

In [84]:
tokenspen = compute_tokens(abbel_objectives_dfs_penalty, 'ABBEL', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [85]:
tokenspen05 = compute_tokens(abbel_objectives_dfs_penalty05, 'ABBEL', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [86]:
tokens01 = compute_tokens(abbel_objectives_dfs01, 'ABBEL', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [40]:
# Create separate legend figure
legend_fig = go.Figure()

# Add invisible traces to create legend
legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='green'),
    name='ABBEL'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='palegreen'),
    name='ABBEL-Length-Penalty-0.1'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='yellowgreen'),
    name='ABBEL-Length-Penalty-0.05'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='orange'),
    name='MEM1-Instruct'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='red'),
    name='MEM1'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='blue'),
    name='Qwen2.5-14B-Instruct'
))

legend_fig.update_layout(
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.5,
        xanchor="center",
        x=0.5,
        font=dict(family='Times New Roman', size=16, color='black'),
        itemsizing='constant'
    ),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    width=800,
    height=100,
    margin=dict(l=0, r=0, t=0, b=0)
)

legend_fig.show()
legend_fig.write_image("figures/QA_legend.pdf", width=800, height=100)

In [192]:
for tokens, scores in zip([tokens01, tokenspen, tokenspen05, mtokens], [scores01, scorespen, scorespen05, mscores]):
    print([str(round(np.mean(s), 3))+'±' + str(round(np.std(s)/np.sqrt(len(s)), 3)) for s in [scores[1], scores[3], scores[4]]])
    print([str(round(np.mean(t), 3))+'±' + str(round(np.std(t)/np.sqrt(len(t)), 3)) for t in [tokens[1], tokens[3], tokens[4]]])
    print()

['0.682±0.031', '2.092±0.077', '3.232±0.125']
['681.206±5.194', '901.855±8.805', '1033.268±14.249']

['0.613±0.043', '1.832±0.111', '2.414±0.156']
['622.921±6.054', '804.077±10.702', '960.141±12.403']

['0.699±0.032', '1.67±0.082', '2.273±0.135']
['683.221±6.184', '922.486±10.617', '1079.409±18.718']



In [99]:
def print_stats(tokens, scores):
    stat_str = ""
    for num_objectives in [1,3,4]: # 2, 8 and 16 objectives
        stat_str += '& ' + str(round(np.mean(scores[num_objectives]), 3))+'$\pm $' + str(round(np.std(scores[num_objectives])/np.sqrt(len(scores[num_objectives])), 3))
        stat_str += " "
        stat_str += '& ' + str(round(np.mean(np.array(tokens[num_objectives])*0.01), 2))+'$\pm $' + str(round(np.std(np.array(tokens[num_objectives])*0.01)/np.sqrt(len(tokens[num_objectives])), 2))
        stat_str += " "
    return stat_str

for tokens, scores in zip([mtokens, tokens01, tokenspen05, tokenspen], [mscores, scores01, scorespen05, scorespen]):
    print(print_stats(tokens, scores))

& 0.788$\pm $0.005 & 6.69$\pm $0.01 & 1.88$\pm $0.024 & 9.13$\pm $0.03 & 2.504$\pm $0.056 & 10.58$\pm $0.07 
& 0.733$\pm $0.005 & 6.78$\pm $0.01 & 2.402$\pm $0.023 & 8.95$\pm $0.03 & 3.569$\pm $0.056 & 10.12$\pm $0.06 
& 0.576$\pm $0.004 & 6.26$\pm $0.01 & 1.733$\pm $0.021 & 7.43$\pm $0.02 & 2.723$\pm $0.049 & 7.75$\pm $0.04 
& 0.52$\pm $0.004 & 6.07$\pm $0.01 & 1.295$\pm $0.02 & 6.59$\pm $0.02 & 1.877$\pm $0.046 & 6.71$\pm $0.03 


In [50]:
# First plot - Bar chart for tokens

fig = go.Figure()

# ABBEL Tokens replace with tokens01
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] - 0.3 for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokens01],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokens01]),
    name='ABBEL',
    marker_color='green',
    width=0.12
))

# ABBEL (Length Penalty) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] - 0.15 for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokenspen05],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokenspen05]),
    name='ABBEL (Length Penalty 0.05)',
    marker_color='yellowgreen',
    width=0.12
))

fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokenspen],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokenspen]),
    name='ABBEL (Length Penalty 0.1)',
    marker_color='palegreen',
    width=0.12
))

# MEM1 (Instruct) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.15for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in mtokens],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in mtokens]),
    name='MEM1 (Instruct)',
    marker_color='orange',
    width=0.12
))

# MEM1 (Reported) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.3 for o in REPORTED_OBJS],
    y=[640,801,1040],
    error_y=dict(type='data', array=[2,6,9]),
    name='MEM1',
    marker_color='red',
    width=0.12
))

# qwen 14b (Reported) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.45 for o in REPORTED_OBJS],
    y=[1560,4470,3840],
    error_y=dict(type='data', array=[19,37,71]),
    name='Qwen2.5-14B-Instruct',
    marker_color='blue',
    width=0.12
))

fig.update_layout(
    xaxis_title='Number of Objectives',
    yaxis_title='Peak Token Usage',
    template='plotly_white',
    font=dict(family='Times New Roman', size=16, color='black'),
    width=500,
    height=400,
    showlegend=False,
    xaxis=dict(tickvals=[OBJS_TO_X[o] for o in ALL_NUM_OBJS], ticktext=ALL_NUM_OBJS, range=[0, 9]),
    yaxis=dict(type='log', tickvals=[500,600,700,800,1000,1200,1500,3000,4500]),
    margin=dict(l=0, r=0, t=0, b=60)
)

fig.show()
fig.write_image("notebooks/figures/QA_peak_tokens.pdf", width=500, height=400)


In [87]:
# First plot - Bar chart for tokens

fig = go.Figure()

# ABBEL Tokens replace with tokens01
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] - 0.3 for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokens01],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokens01]),
    name='ABBEL',
    marker_color='green',
    width=0.12
))

# ABBEL (Length Penalty) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] - 0.15 for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokenspen05],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokenspen05]),
    name='ABBEL (Length Penalty 0.05)',
    marker_color='yellowgreen',
    width=0.12
))

fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokenspen],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokenspen]),
    name='ABBEL (Length Penalty 0.1)',
    marker_color='palegreen',
    width=0.12
))

# MEM1 (Instruct) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.15for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in mtokens],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in mtokens]),
    name='MEM1 (Instruct)',
    marker_color='orange',
    width=0.12
))

# MEM1 (Reported) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.3 for o in REPORTED_OBJS],
    y=[640,801,1040],
    error_y=dict(type='data', array=[2,6,9]),
    name='MEM1',
    marker_color='red',
    width=0.12
))

# qwen 14b (Reported) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.45 for o in REPORTED_OBJS],
    y=[1560,4470,3840],
    error_y=dict(type='data', array=[19,37,71]),
    name='Qwen2.5-14B-Instruct',
    marker_color='blue',
    width=0.12
))

fig.update_layout(
    xaxis_title='Number of Objectives',
    yaxis_title='Peak Token Usage',
    template='plotly_white',
    font=dict(family='Times New Roman', size=16, color='black'),
    width=500,
    height=400,
    showlegend=False,
    xaxis=dict(tickvals=[OBJS_TO_X[o] for o in ALL_NUM_OBJS], ticktext=ALL_NUM_OBJS, range=[0, 9]),
    yaxis=dict(type='log', tickvals=[500,600,700,800,1000,1200,1500,3000,4500]),
    margin=dict(l=0, r=0, t=0, b=60)
)

fig.show()
fig.write_image("notebooks/figures/QA_peak_tokens_fulleval.pdf", width=500, height=400)


In [70]:
internal_states = compute_tokens(mem1_objectives_dfs, 'MEM1', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
missing </information>
missing opening and/or closing think tags
peak tokens for  4 objs complete
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
peak tokens for  8 objs complete
peak 

In [71]:
beliefs = compute_tokens(abbel_objectives_dfs01, 'ABBEL', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [72]:
beliefs_pen = compute_tokens(abbel_objectives_dfs_penalty, 'ABBEL', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [73]:
beliefs_pen05 = compute_tokens(abbel_objectives_dfs_penalty05, 'ABBEL', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [ ]:
fig = go.Figure()

# ABBEL Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] - 0.3 for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in beliefs],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in beliefs]),
    name='ABBEL',
    marker_color='green',
    width=0.15
))

# ABBEL (Length Penalty) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] -0.1 for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in beliefs_pen05],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in beliefs_pen05]),
    name='ABBEL (Length Penalty 0.1)',
    marker_color='yellowgreen',
    width=0.15
))

# ABBEL (Length Penalty) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.1for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in beliefs_pen],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in beliefs_pen]),
    name='ABBEL (Length Penalty 0.1)',
    marker_color='palegreen',
    width=0.15
))

# MEM1 (Instruct) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + 0.3 for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in internal_states],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in internal_states]),
    name='MEM1 (Instruct)',
    marker_color='orange',
    width=0.15
))

fig.update_layout(
    xaxis_title='Number of Objectives',
    yaxis_title='Peak Internal State Tokens',
    template='plotly_white',
    font=dict(family='Times New Roman', size=16, color='black'),
    width=500,
    height=400,
    showlegend=False,
    xaxis=dict(tickvals=[OBJS_TO_X[o] for o in ALL_NUM_OBJS], ticktext=ALL_NUM_OBJS, range=[0, 9]),
    margin=dict(l=0, r=0, t=0, b=60)
)

fig.show()
#fig.write_image("figures/QA_peak_tokens_memory_only.pdf", width=500, height=400)
fig.write_image("notebooks/figures/QA_peak_tokens_memory_only_fulleval.pdf", width=500, height=400)

In [89]:
scores01 = compute_scores(abbel_objectives_dfs01)
scorespen = compute_scores(abbel_objectives_dfs_penalty)
scorespen05 = compute_scores(abbel_objectives_dfs_penalty05)
mscores = compute_scores(mem1_objectives_dfs)

# Second plot - Line chart for exact match scores
fig2 = go.Figure()

# ABBEL EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in scores01],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in scores01]),
    mode='markers+lines',
    name='ABBEL',
    marker_color='green',
    marker_symbol='circle'
))

# ABBEL penalty EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in scorespen],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in scorespen]),
    mode='markers+lines',
    name='ABBEL (Length Penalty 0.1)',
    marker_color='palegreen',
    marker_symbol='circle'
))

fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in scorespen05],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in scorespen05]),
    mode='markers+lines',
    name='ABBEL (Length Penalty 0.05)',
    marker_color='yellowgreen',
    marker_symbol='circle'
))

# MEM1 (Instruct) EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in mscores],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in mscores]),
    mode='markers+lines',
    name='MEM1 (Instruct)',
    marker_color='orange',
    marker_symbol='circle'
))

# MEM1 (Reported) EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in REPORTED_OBJS],
    y=[0.709,1.87,1.97],
    mode='markers',
    name='MEM1',
    marker_color='red',
    marker_symbol='circle',
    marker_size=10
))

# Qwen 14B
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in REPORTED_OBJS],
    y=[0.732,1.55,0.567],
    mode='markers',
    name='Qwen2.5-14B-Instruct',
    marker_color='blue',
    marker_symbol='circle',
    marker_size=10
))

fig2.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=16, color='black'),
    xaxis_title='Number of Objectives',
    yaxis_title='Exact Match Count',
    width=500,
    height=400,
    showlegend=False,
    xaxis=dict(tickvals=[0] + [OBJS_TO_X[o] for o in ALL_NUM_OBJS], ticktext=[0] + ALL_NUM_OBJS, range=[0, 9]),
    margin=dict(l=0, r=0, t=0, b=60)
)

fig2.show()
fig2.write_image("notebooks/figures/QA_scores_fulleval.pdf", width=500, height=400)
